# Runbook 3.1.d: Packet capture and traffic sniffing  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 3.1.B, 3.1.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You are triaging a captured packet log at Crestfield Analytics. For each entry, determine whether
it exposes a plaintext credential ("LEAK") or not ("SAFE"):

- **Encrypted ports are always "SAFE"**: regardless of payload content: port `443` (HTTPS) and
  port `22` (SSH) encrypt their payload, so nothing in the payload text can be treated as an
  actual leaked credential.
- **Cleartext-capable ports** are `80` (HTTP), `23` (Telnet), and `21` (FTP). An entry on one of
  these ports is **"LEAK"** if its payload contains any credential indicator string (case-sensitive
  substring match), otherwise it is **"SAFE"**.
- Credential indicator strings to check for (substring match anywhere in the payload):
  `"password="`, `"Password:"`, `"PASS "`.
- Any other port not listed above is treated as `"SAFE"` (out of scope for this detector).

**Lesson objective.** Explain how packet capture tools intercept network traffic, apply capture and display filters to isolate relevant packets, and analyze a small packet log to identify credentials or indicators of compromise sent in cleartext.

### What comes in
- `packets`: `list[dict]`, each with:
  - `"port"`: `int`, destination port
  - `"payload"`: `str`, the captured payload text

### What must come out
A `list[str]` of classifications, **one per input packet, in the same order as the input**. Each
entry is one of `"LEAK"`, `"SAFE"`.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
packets = [
    {"port": 80, "payload": "POST /login HTTP/1.1\r\n\r\nuser=alice&password=Pass123!"},
    {"port": 443, "payload": "\\x16\\x03\\x03 encrypted application data"},
]
solve(packets)  ->  ["LEAK", "SAFE"]

packets2 = [
    {"port": 23, "payload": "login: admin\r\nPassword: R0uterAdmin!"},
    {"port": 80, "payload": "GET /index.html HTTP/1.1"},
]
solve(packets2)  ->  ["LEAK", "SAFE"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Port check first: if `port` is `443` or `22`, the result is always `"SAFE"`.
- [ ] Otherwise, if `port` is `80`, `23`, or `21`: check for any of `"password="`, `"Password:"`,
- [ ] Any other port: always `"SAFE"`.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(packets: list[dict]) -> list[str]
```

#### Input (`packets`)
`list[dict]` where each dict has:
- `"port"`: `int`, destination port
- `"payload"`: `str`, captured payload text

#### Output (`result`)
A `list[str]` the same length as `packets`, in the **same order as the input**, where each element
is one of `"LEAK"`, `"SAFE"`.

#### Classification rules
1. If `port` is `443` or `22`, result is `"SAFE"` (encrypted; payload content is never checked).
2. Else if `port` is `80`, `23`, or `21`: result is `"LEAK"` if `payload` contains any of the
   substrings `"password="`, `"Password:"`, `"PASS "`; otherwise `"SAFE"`.
3. Else (any other port), result is `"SAFE"`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output order matches input order exactly.
- Substring checks are case-sensitive exact matches of the three literal strings above.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Dict

_ENCRYPTED_PORTS = {443, 22}
_CLEARTEXT_PORTS = {80, 23, 21}
_INDICATORS = ("password=", "Password:", "PASS ")


def solve(packets: List[Dict]) -> List[str]:
    result = []
    for p in packets:
        port = p["port"]
        if port in _ENCRYPTED_PORTS:
            result.append("SAFE")
        elif port in _CLEARTEXT_PORTS:
            payload = p["payload"]
            leaked = any(ind in payload for ind in _INDICATORS)
            result.append("LEAK" if leaked else "SAFE")
        else:
            result.append("SAFE")
    return result


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def _p(port, payload):
    return {"port": port, "payload": payload}


def test_http_login_leak():
    packets = [_p(80, "POST /login HTTP/1.1\r\n\r\nuser=alice&password=Pass123!")]
    assert impl.solve(packets) == ["LEAK"]


def test_https_always_safe():
    packets = [_p(443, "password= this looks like a leak but is encrypted TLS payload")]
    assert impl.solve(packets) == ["SAFE"]


def test_telnet_password_leak():
    packets = [_p(23, "login: admin\r\nPassword: R0uterAdmin!")]
    assert impl.solve(packets) == ["LEAK"]


def test_http_no_credential_safe():
    packets = [_p(80, "GET /index.html HTTP/1.1")]
    assert impl.solve(packets) == ["SAFE"]


def test_ftp_pass_leak():
    packets = [_p(21, "USER backup\r\nPASS Backup#2024")]
    assert impl.solve(packets) == ["LEAK"]


def test_ssh_always_safe():
    packets = [_p(22, "SSH-2.0-OpenSSH_9.3 key exchange")]
    assert impl.solve(packets) == ["SAFE"]


def test_out_of_scope_port_safe():
    packets = [_p(53, "DNS query: mail.example A?")]
    assert impl.solve(packets) == ["SAFE"]


def test_empty_packets():
    assert impl.solve([]) == []


def test_mixed_batch_order_preserved():
    packets = [
        _p(80, "password=leaked1"),
        _p(443, "password=leaked2"),
        _p(21, "PASS ftpsecret"),
    ]
    assert impl.solve(packets) == ["LEAK", "SAFE", "LEAK"]


def test_case_sensitivity_lowercase_pass_not_matched():
    packets = [_p(21, "this text says pass phrase but not the exact marker")]
    assert impl.solve(packets) == ["SAFE"]


def test_https_with_literal_password_text_still_safe():
    packets = [_p(443, "Password: totally-visible-but-port-is-encrypted")]
    assert impl.solve(packets) == ["SAFE"]


def test_multiple_indicators_still_single_leak():
    packets = [_p(80, "password=abc\r\nPassword: def\r\nPASS ghi")]
    assert impl.solve(packets) == ["LEAK"]


def test_unrelated_cleartext_port_content_safe():
    packets = [_p(23, "just some banner text, no credentials here")]
    assert impl.solve(packets) == ["SAFE"]


def test_large_batch_deterministic():
    packets = []
    expected = []
    for i in range(15):
        if i % 3 == 0:
            packets.append(_p(80, f"password=leak{i}"))
            expected.append("LEAK")
        elif i % 3 == 1:
            packets.append(_p(443, f"password=leak{i}"))
            expected.append("SAFE")
        else:
            packets.append(_p(9999, f"password=leak{i}"))
            expected.append("SAFE")
    assert impl.solve(packets) == expected


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it